In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb

# Grouping by day

In [ ]:
# Initialize DuckDB connection
con = duckdb.connect()

# Load join dates (small table, can keep in memory)
user_of_interests = "../data/user_activity/filtered/first_week_blockers.parquet"
con.execute("CREATE TABLE join_dates AS SELECT * FROM read_parquet(?)", [user_of_interests])

print("Join dates loaded")

Join dates loaded:


,count
0,100000


In [ ]:
# Helper: build 7-day vectors using DuckDB for efficient large-table processing
def make_vec_duckdb(file_path, left_on='did_id', vec_name='vec'):
    """
    Use DuckDB to:
    1. Read parquet file
    2. Join with join_dates
    3. Compute days_since_join
    4. Filter to first week (0-6 days)
    5. Aggregate counts per user per day
    6. Pivot to create 7-element vectors
    """
    query = f"""
    WITH activity AS (
        SELECT 
            act.{left_on} as id_col,
            act.created_date,
            jd.join_date
        FROM read_parquet('{file_path}') act
        LEFT JOIN join_dates jd ON act.{left_on} = jd.did_id
    ),
    days_computed AS (
        SELECT 
            id_col,
            CAST(ROUND((EPOCH(created_date::TIMESTAMP) - EPOCH(join_date::TIMESTAMP)) / 86400.0) AS INTEGER) AS days_since_join
        FROM activity
        WHERE join_date IS NOT NULL
    ),
    first_week AS (
        SELECT id_col, days_since_join
        FROM days_computed
        WHERE days_since_join >= 0 AND days_since_join <= 6
    ),
    counts AS (
        SELECT 
            id_col,
            days_since_join,
            COUNT(*) as cnt
        FROM first_week
        GROUP BY id_col, days_since_join
    ),
    pivoted AS (
        SELECT
            id_col,
            MAX(CASE WHEN days_since_join = 0 THEN cnt ELSE 0 END) AS d0,
            MAX(CASE WHEN days_since_join = 1 THEN cnt ELSE 0 END) AS d1,
            MAX(CASE WHEN days_since_join = 2 THEN cnt ELSE 0 END) AS d2,
            MAX(CASE WHEN days_since_join = 3 THEN cnt ELSE 0 END) AS d3,
            MAX(CASE WHEN days_since_join = 4 THEN cnt ELSE 0 END) AS d4,
            MAX(CASE WHEN days_since_join = 5 THEN cnt ELSE 0 END) AS d5,
            MAX(CASE WHEN days_since_join = 6 THEN cnt ELSE 0 END) AS d6
        FROM counts
        GROUP BY id_col
    )
    SELECT 
        id_col AS did_id,
        [d0, d1, d2, d3, d4, d5, d6] AS {vec_name}
    FROM pivoted
    """
    
    return con.execute(query).df()

print("DuckDB helper function defined")

DuckDB helper function defined


In [7]:
# Posts vectors using DuckDB
posts_path = "../data/user_activity/filtered/posts.parquet"
posts_vec_df = make_vec_duckdb(posts_path, left_on='did_id', vec_name='posts_vec')

# Start building the result table
con.execute("CREATE TABLE uoi_activity AS SELECT jd.*, pv.posts_vec FROM join_dates jd LEFT JOIN posts_vec_df pv ON jd.did_id = pv.did_id")

# Fill nulls with [0]*7
result = con.execute("""
    SELECT 
        *,
        COALESCE(posts_vec, [0,0,0,0,0,0,0]) AS posts_vec_fixed
    FROM uoi_activity
""").df()

con.execute("DROP TABLE uoi_activity")
con.execute("CREATE TABLE uoi_activity AS SELECT * EXCLUDE (posts_vec), posts_vec_fixed AS posts_vec FROM result")

print("Posts processed:")
print(con.execute("SELECT did_id, posts_vec FROM uoi_activity LIMIT 5").df())

BinderException: Binder Error: Ambiguous reference to column name "did_id" (use: "act.did_id" or "jd.did_id")

In [ ]:
# Blocks: actor and subject 7-day vectors using DuckDB
blocks_path = "../data/user_activity/filtered/blocks.parquet"

# Actor vectors
blocks_actor_vec = make_vec_duckdb(blocks_path, left_on='did_id', vec_name='blocks_actor_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(bav.blocks_actor_vec, [0,0,0,0,0,0,0]) AS blocks_actor_vec
    FROM uoi_activity ua
    LEFT JOIN blocks_actor_vec bav ON ua.did_id = bav.did_id
""")

# Subject vectors
blocks_subject_vec = make_vec_duckdb(blocks_path, left_on='subject_did_id', vec_name='blocks_subject_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(bsv.blocks_subject_vec, [0,0,0,0,0,0,0]) AS blocks_subject_vec
    FROM uoi_activity ua
    LEFT JOIN blocks_subject_vec bsv ON ua.did_id = bsv.did_id
""")

print("Blocks processed:")
print(con.execute("SELECT did_id, blocks_actor_vec, blocks_subject_vec FROM uoi_activity LIMIT 5").df())

Initial blocks count: 2199771
     did_id  join_date                   posts_vec       blocks_actor_vec  \
0  14530036 2024-04-13       [0, 3, 1, 0, 0, 0, 0]  [4, 1, 0, 0, 0, 0, 0]   
1   4701670 2024-11-13  [3, 9, 23, 46, 81, 40, 20]  [0, 0, 1, 1, 2, 3, 0]   
2  27369031 2024-11-15   [11, 6, 17, 9, 19, 13, 9]  [0, 0, 1, 0, 5, 0, 0]   
3  14811432 2024-10-17       [0, 0, 1, 0, 0, 0, 0]  [0, 0, 1, 0, 0, 0, 0]   
4  14747413 2024-10-17      [9, 34, 8, 3, 0, 4, 1]  [2, 1, 0, 0, 0, 0, 0]   

      blocks_subject_vec  
0  [0, 0, 0, 0, 0, 0, 0]  
1  [0, 0, 0, 0, 0, 0, 0]  
2  [0, 0, 0, 0, 0, 0, 0]  
3  [0, 0, 0, 0, 0, 0, 0]  
4  [0, 0, 0, 0, 0, 0, 0]  


In [ ]:
# Follows: actor and subject vectors using DuckDB (handles large tables efficiently)
follows_path = "../data/user_activity/filtered/follows.parquet"

# Actor vectors
follows_actor_vec = make_vec_duckdb(follows_path, left_on='did_id', vec_name='follows_actor_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(fav.follows_actor_vec, [0,0,0,0,0,0,0]) AS follows_actor_vec
    FROM uoi_activity ua
    LEFT JOIN follows_actor_vec fav ON ua.did_id = fav.did_id
""")

# Subject vectors
follows_subject_vec = make_vec_duckdb(follows_path, left_on='subject_did_id', vec_name='follows_subject_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(fsv.follows_subject_vec, [0,0,0,0,0,0,0]) AS follows_subject_vec
    FROM uoi_activity ua
    LEFT JOIN follows_subject_vec fsv ON ua.did_id = fsv.did_id
""")

print("Follows processed:")
print(con.execute("SELECT did_id, follows_actor_vec, follows_subject_vec FROM uoi_activity LIMIT 5").df())

: 

In [ ]:
# Likes: actor and subject vectors using DuckDB
likes_path = "../data/user_activity/filtered/likes.parquet"

# Actor vectors
likes_actor_vec = make_vec_duckdb(likes_path, left_on='did_id', vec_name='likes_actor_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(lav.likes_actor_vec, [0,0,0,0,0,0,0]) AS likes_actor_vec
    FROM uoi_activity ua
    LEFT JOIN likes_actor_vec lav ON ua.did_id = lav.did_id
""")

# Subject vectors
likes_subject_vec = make_vec_duckdb(likes_path, left_on='subject_did_id', vec_name='likes_subject_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(lsv.likes_subject_vec, [0,0,0,0,0,0,0]) AS likes_subject_vec
    FROM uoi_activity ua
    LEFT JOIN likes_subject_vec lsv ON ua.did_id = lsv.did_id
""")

print("Likes processed:")
print(con.execute("SELECT did_id, likes_actor_vec, likes_subject_vec FROM uoi_activity LIMIT 5").df())

Initial likes count: 9854959
     did_id                    likes_actor_vec             likes_subject_vec
0  18552585  [1258, 1285, 225, 305, 69, 71, 0]  [332, 92, 52, 46, 28, 32, 5]
1  16709725            [2, 29, 5, 8, 0, 0, 14]         [1, 2, 2, 2, 0, 0, 0]
2   2645420       [17, 97, 32, 49, 19, 37, 16]    [2, 14, 7, 11, 15, 15, 19]
3  34159973       [18, 21, 18, 27, 26, 35, 24]  [24, 15, 12, 11, 12, 18, 20]
4  18746738             [8, 22, 0, 0, 0, 0, 0]         [2, 7, 0, 0, 0, 0, 0]


## Save the file

In [ ]:
# Export final result to parquet using DuckDB
save_path = "../data/user_activity/processed/user_activity.parquet"

# Get final dataframe and save
final_df = con.execute("SELECT * FROM uoi_activity").df()

# Convert to pyarrow and save (DuckDB list types are compatible with pyarrow)
arrow_table = pa.Table.from_pandas(final_df, preserve_index=False)
pq.write_table(arrow_table, save_path, compression='zstd')

print('Saved user_table to', save_path)
print('Number of rows:', len(final_df))
print('Columns:', list(final_df.columns))
print('\nSample of final data:')
print(final_df.head())

# Close DuckDB connection
con.close()

Saved user_table to ../data/posting/processed/user_activity.parquet
Vector columns saved: ['posts_vec', 'blocks_actor_vec', 'blocks_subject_vec', 'follows_actor_vec', 'follows_subject_vec', 'likes_actor_vec', 'likes_subject_vec']
Number of rows: 52650
